# Notebook 02 — DEFT + Fusion: TRAIN on seeds, then extract features

Two phases:
1. **Train** DEFT (STN+Gaussian) + a **trainable** Cross-Modal Co-Attention fusion + a small classifier head on the **10% seed frames** (multi-label Asymmetric Loss). Backbones stay frozen. This makes DEFT actually focus on the hand-object region instead of sitting at identity, and gives the fusion meaningful weights.
2. **Extract** 512-D features for all frames with the trained DEFT+fusion (head discarded) -> `<P>_feats.npz` for NB03.

> Needs the dataset on disk + a GPU. Run NB01 first (it writes the seeds/targets this trains on).

In [16]:
# ===== CONFIG =====
import torch, torch.nn as nn, torch.nn.functional as Fnn
import torchvision as tv
from torchvision import transforms
from PIL import Image
from pathlib import Path
import numpy as np, pickle
try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x,**k): return x

DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
OUT_DIR=Path('./artifacts')
IMG_SIZE=224; FEAT_DIM=512
WARP_FLOW=False

# training knobs
EPOCHS=15; BATCH=32; LR=1e-3; LOSS='asl'   # 'asl' (asymmetric) | 'bce'
FINETUNE_BLOCK='layer4'                     # unfreeze this ResNet block on the seeds ('layer4' | None)
BACKBONE_LR=1e-4                            # smaller LR for the unfrozen backbone block
DEFT_LR=5e-3                                # higher LR for DEFT (its gradient is weakest in the chain)
FORCE_REEXTRACT=True                        # features change after training -> recompute

ADL_ROOT=Path(r'D:/Datasets/Datasets/ADL')
PARTICIPANTS=['P_10', 'P_11']
def rgb_dir(p):  return ADL_ROOT/'Frames'/p/f'Original_{p}'
def flow_dir(p): return ADL_ROOT/'OpticalFlow'/p/'viz'
print('device:',DEVICE,'| epochs',EPOCHS,'| loss',LOSS)

device: cuda | epochs 15 | loss asl


## DEFT (STN + Gaussian) — trainable

In [17]:
class DEFT(nn.Module):
    def __init__(self,sigma_g=0.5,center=(0.0,0.0)):
        super().__init__()
        self.loc=nn.Sequential(nn.Conv2d(3,8,7),nn.ReLU(True),nn.MaxPool2d(2,2),
                               nn.Conv2d(8,10,5),nn.ReLU(True),nn.MaxPool2d(2,2))
        self.fc=None; self.sigma_g=sigma_g
        self.register_buffer('center',torch.tensor(center).float())
        # per-param allowed deviation from identity: [a,b,tx, c,d,ty]
        self.register_buffer('base',torch.tensor([1,0,0,0,1,0.]))
        self.register_buffer('rng_',torch.tensor([0.3,0.3,0.5,0.3,0.3,0.5]))
    def _fc(self,flat):
        self.fc=nn.Sequential(nn.Linear(flat,32),nn.ReLU(True),nn.Linear(32,6)).to(self.center.device)
        nn.init.normal_(self.fc[-1].weight,std=1e-2)   # small random -> starts slightly off identity
        self.fc[-1].bias.data.zero_()                  # bias 0 -> bounded theta starts AT identity
    def mask(self,B,H,W,dev):
        ys=torch.linspace(-1,1,H,device=dev); xs=torch.linspace(-1,1,W,device=dev)
        gy,gx=torch.meshgrid(ys,xs,indexing='ij'); cx,cy=self.center
        g=torch.exp(-((gx-cx)**2+(gy-cy)**2)/(2*self.sigma_g**2)); return g.view(1,1,H,W).expand(B,1,H,W)
    def theta(self,x):
        f=self.loc(x).flatten(1)
        if self.fc is None: self._fc(f.shape[1])
        delta=torch.tanh(self.fc(f))                  # (-1,1), bounded
        th=self.base+self.rng_*delta                  # identity +/- allowed range
        return th.view(-1,2,3)
    def warp(self,x,th): return Fnn.grid_sample(x,Fnn.affine_grid(th,x.size(),align_corners=False),align_corners=False)
    def forward(self,rgb,flow=None):
        th=self.theta(rgb); rw=self.warp(rgb,th)
        g=self.mask(rw.size(0),rw.size(2),rw.size(3),rw.device); rgb_out=rw*g
        flow_out=flow
        if WARP_FLOW and flow is not None: flow_out=self.warp(flow,th)*g
        return rgb_out,flow_out

## Frozen ResNet-18 backbones + trainable Co-Attention fusion + classifier head
The fusion now keeps a learnable `Linear(1024->512)` — legitimate because it will be **trained** (the earlier problem was that this matrix was random/untrained). The head is a linear multi-label classifier used only to supervise feature learning; NB03 does not use it.

In [11]:
def make_backbone():
    m=tv.models.resnet18(weights=tv.models.ResNet18_Weights.IMAGENET1K_V1); m.fc=nn.Identity()
    for p in m.parameters(): p.requires_grad=False
    if FINETUNE_BLOCK and hasattr(m,FINETUNE_BLOCK):
        for p in getattr(m,FINETUNE_BLOCK).parameters(): p.requires_grad=True   # unfreeze last block
    return m.eval().to(DEVICE)   # eval() keeps BatchNorm running-stats frozen even while layer4 trains
backbone_rgb=make_backbone(); backbone_flow=make_backbone()

class CoAttnFusion(nn.Module):
    def __init__(self,d=512):
        super().__init__(); self.proj=nn.Linear(2*d,d)
    def forward(self,frgb,fflow):
        a=(Fnn.normalize(frgb,dim=1)*Fnn.normalize(fflow,dim=1)).sum(1,keepdim=True)
        r_rgb=frgb+a*fflow; r_flow=fflow+a*frgb
        return self.proj(torch.cat([r_rgb,r_flow],dim=1))

deft=DEFT().to(DEVICE)
fusion=CoAttnFusion(FEAT_DIM).to(DEVICE)
CMAP=pickle.load(open(OUT_DIR/'class_map.pkl','rb')); C=CMAP['C']
head=nn.Linear(FEAT_DIM,C).to(DEVICE)
def trainable_backbone_params():
    return [q for bb in (backbone_rgb,backbone_flow) for q in bb.parameters() if q.requires_grad]
nb_bp=sum(q.numel() for q in trainable_backbone_params())
print(f'modules ready | C = {C} | finetune={FINETUNE_BLOCK} | backbone trainable params={nb_bp:,}')

modules ready | C = 13 | finetune=layer4 | backbone trainable params=16,787,456


## Asymmetric Loss (multi-label, handles class imbalance)

In [12]:
class ASL(nn.Module):
    def __init__(self,gamma_neg=4,gamma_pos=1,clip=0.05,eps=1e-8):
        super().__init__(); self.gn,self.gp,self.clip,self.eps=gamma_neg,gamma_pos,clip,eps
    def forward(self,logits,y):
        xs_pos=torch.sigmoid(logits); xs_neg=1-xs_pos
        if self.clip>0: xs_neg=(xs_neg+self.clip).clamp(max=1)
        los_pos=y*torch.log(xs_pos.clamp(min=self.eps))
        los_neg=(1-y)*torch.log(xs_neg.clamp(min=self.eps))
        loss=los_pos+los_neg
        pt=xs_pos*y+xs_neg*(1-y); gamma=self.gp*y+self.gn*(1-y)
        loss*=(1-pt)**gamma
        return -loss.sum()/y.shape[0]
criterion=ASL() if LOSS=='asl' else nn.BCEWithLogitsLoss()
print('loss =',LOSS)

loss = asl


## Build the pooled seed training set (all participants' seed frames)

In [13]:
tf=transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)),transforms.ToTensor(),
                       transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
def load_img(path): return tf(Image.open(path).convert('RGB')).unsqueeze(0)
def load_batch(p,idxs):
    rd,fd=rgb_dir(p),flow_dir(p)
    rgb=torch.cat([load_img(rd/f'frame_{i:05d}.jpg') for i in idxs],0).to(DEVICE)
    flow=torch.cat([load_img(fd/f'frame_{i:05d}.jpg') for i in idxs],0).to(DEVICE)
    return rgb,flow

# pool (participant, frame_idx, multihot) over the seed masks from NB01
train_items=[]; TARG={}
for p in PARTICIPANTS:
    d=np.load(OUT_DIR/f'{p}_data.npz',allow_pickle=True); Y,seeds=d['targets'],d['seeds']; TARG[p]=Y
    for i in np.where(seeds)[0]: train_items.append((p,int(i)))
print(f'training on {len(train_items)} seed frames from {PARTICIPANTS}')
ck_y=np.stack([TARG[p][i] for p,i in train_items])
print(f'  [check] seed labels: shape={ck_y.shape} pos/frame mean={ck_y.sum(1).mean():.2f} class coverage={(ck_y.sum(0)>0).sum()}/{C}')

training on 3580 seed frames from ['P_10', 'P_11']
  [check] seed labels: shape=(3580, 13) pos/frame mean=1.08 class coverage=13/13


## Train (DEFT + fusion + head); backbones frozen

In [14]:
rng=np.random.default_rng(0)
# warmup forward on ONE real seed frame to build DEFT.fc lazily, so its params join the optimizer
with torch.no_grad():
    p0,i0=train_items[0]; r,_=load_batch(p0,[i0]); deft.theta(r)
deft_params=[q for q in deft.parameters() if q.requires_grad]
hf_params=[q for q in (list(fusion.parameters())+list(head.parameters())) if q.requires_grad]
bb_params=trainable_backbone_params()                  # unfrozen backbone block (layer4)
opt=torch.optim.Adam([{'params':deft_params,'lr':DEFT_LR},
                      {'params':hf_params,'lr':LR},
                      {'params':bb_params,'lr':BACKBONE_LR}],weight_decay=1e-4)
params=deft_params+hf_params+bb_params
print(f'trainable tensors: {len(params)} | trainable params: {sum(q.numel() for q in params):,}')

deft.train(); fusion.train(); head.train()
items=np.array(train_items,dtype=object)
for epoch in range(EPOCHS):
    perm=rng.permutation(len(items)); tot=0.0
    pbar=tqdm(range(0,len(perm),BATCH),desc=f'epoch {epoch+1}/{EPOCHS}')
    for b in pbar:
        sel=items[perm[b:b+BATCH]]
        # group by participant for correct paths
        rgb_l=[]; flow_l=[]; y_l=[]
        for p,i in sel:
            rgb_l.append((p,int(i))); y_l.append(TARG[p][int(i)])
        # batch load (mixed participants ok: load individually then cat)
        rgb=torch.cat([load_img(rgb_dir(p)/f'frame_{i:05d}.jpg') for p,i in rgb_l],0).to(DEVICE)
        flow=torch.cat([load_img(flow_dir(p)/f'frame_{i:05d}.jpg') for p,i in rgb_l],0).to(DEVICE)
        yb=torch.tensor(np.stack(y_l),dtype=torch.float32,device=DEVICE)
        rgb_d,flow_d=deft(rgb,flow)
        frgb=backbone_rgb(rgb_d); fflow=backbone_flow(flow_d if WARP_FLOW else flow)
        logits=head(fusion(frgb,fflow))
        loss=criterion(logits,yb)
        opt.zero_grad(); loss.backward(); opt.step()
        tot+=loss.item()*len(sel); pbar.set_postfix(loss=float(loss))
    print(f'  epoch {epoch+1}: mean loss = {tot/len(items):.4f}')
# save trained weights
torch.save({'deft':deft.state_dict(),'fusion':fusion.state_dict()},OUT_DIR/'deft_fusion.pt')
print('saved trained DEFT+fusion -> deft_fusion.pt')

# --- DEFT diagnostic: did the STN move away from identity? ---
deft.eval()
with torch.no_grad():
    sel=items[rng.permutation(len(items))[:min(64,len(items))]]
    rgb=torch.cat([load_img(rgb_dir(p)/f'frame_{int(i):05d}.jpg') for p,i in sel],0).to(DEVICE)
    th=deft.theta(rgb)                       # (B,2,3)
    I=torch.tensor([[1,0,0],[0,1,0.]],device=DEVICE)
    dev=(th-I).abs().mean().item()
print(f'[DEFT diag] mean |theta - identity| = {dev:.4f}  (≈0 means DEFT stayed at identity / did not focus)')

trainable tensors: 42 | trainable params: 18,187,629


epoch 1/15:   0%|          | 0/112 [00:00<?, ?it/s]

  epoch 1: mean loss = 0.2283


epoch 2/15:   0%|          | 0/112 [00:00<?, ?it/s]

  epoch 2: mean loss = 0.0415


...

  epoch 13: mean loss = 0.0002


epoch 14/15:   0%|          | 0/112 [00:00<?, ?it/s]

  epoch 14: mean loss = 0.0001


epoch 15/15:   0%|          | 0/112 [00:00<?, ?it/s]

  epoch 15: mean loss = 0.0000
saved trained DEFT+fusion -> deft_fusion.pt
[DEFT diag] mean |theta - identity| = 0.3656  (≈0 means DEFT stayed at identity / did not focus)


## Extract features for ALL frames with the trained DEFT+fusion (head discarded)

In [15]:
deft.eval(); fusion.eval()
@torch.inference_mode()
def extract(p):
    rd,fd=rgb_dir(p),flow_dir(p); n=len(sorted(rd.glob('frame_*.jpg')))
    feats=np.zeros((n,FEAT_DIM),np.float32)
    for i in tqdm(range(n),desc=f'{p}',unit='f'):
        rgb=load_img(rd/f'frame_{i:05d}.jpg').to(DEVICE); flow=load_img(fd/f'frame_{i:05d}.jpg').to(DEVICE)
        rgb_d,flow_d=deft(rgb,flow)
        frgb=backbone_rgb(rgb_d); fflow=backbone_flow(flow_d if WARP_FLOW else flow)
        if i==0: print(f'  [check] rgb{tuple(rgb.shape)}->DEFT{tuple(rgb_d.shape)}->feat{tuple(frgb.shape)}->fused{tuple(fusion(frgb,fflow).shape)}')
        feats[i]=fusion(frgb,fflow).squeeze(0).cpu().numpy()
    return feats

for p in PARTICIPANTS:
    out=OUT_DIR/f'{p}_feats.npz'
    if out.exists() and not FORCE_REEXTRACT:
        print(f'{p}: cached (skip)'); continue
    feats=extract(p); np.savez_compressed(out,feats=feats)
    print(f'  [check] {p} feats: shape={feats.shape} mean={feats.mean():.3f} std={feats.std():.3f} min={feats.min():.3f} max={feats.max():.3f}')
    print(f'saved -> {out}')

P_10:   0%|          | 0/28674 [00:00<?, ?f/s]

  [check] rgb(1, 3, 224, 224)->DEFT(1, 3, 224, 224)->feat(1, 512)->fused(1, 512)
  [check] P_10 feats: shape=(28674, 512) mean=-0.032 std=1.160 min=-7.891 max=8.179
saved -> artifacts\P_10_feats.npz


P_11:   0%|          | 0/14775 [00:00<?, ?f/s]

  [check] rgb(1, 3, 224, 224)->DEFT(1, 3, 224, 224)->feat(1, 512)->fused(1, 512)
  [check] P_11 feats: shape=(14775, 512) mean=-0.088 std=1.293 min=-6.479 max=6.757
saved -> artifacts\P_11_feats.npz
